# reproduce_results.ipynb

Loads previously trained models and vectorizers from disk, builds features,
runs predictions and reports ROC-AUC, Precision and Recall for every split
(train / validation / test) and both models (baseline and improved).

**This notebook does not train anything.**

Run `train_models.ipynb` first to populate the `models/` directory.

In [6]:
import os, sys
import pandas as pd
import numpy as np
import joblib

sys.path.insert(0, os.path.dirname(os.path.abspath('utils.py')))
from utils import (
    get_data_splits,
    get_bow_features,
    build_all_features,
    evaluate_model,
)

## 1 – Load data & recreate splits

In [7]:
DATA_PATH_DEFAULT = os.path.join(os.path.expanduser("~"), "Datasets", "QuoraQuestionPairs", "quora_data.csv")
DATA_PATH_LOCAL = "quora_data.csv"

if os.path.exists(DATA_PATH_DEFAULT):
    DATA_PATH = DATA_PATH_DEFAULT
elif os.path.exists(DATA_PATH_LOCAL):
    DATA_PATH = DATA_PATH_LOCAL
else:
    raise FileNotFoundError(f"Could not find quora_data.csv in {DATA_PATH_DEFAULT} or {DATA_PATH_LOCAL}")

quora_df = pd.read_csv(DATA_PATH)
train_df, val_df, test_df = get_data_splits(quora_df)

print('train_df.shape=', train_df.shape)
print('val_df.shape=',   val_df.shape)
print('test_df.shape=',  test_df.shape)

train_df.shape= (291897, 6)
val_df.shape= (15363, 6)
test_df.shape= (16172, 6)


## 2 – Load models from disk

In [8]:
MODELS_DIR = "models"

count_vec   = joblib.load(f"{MODELS_DIR}/count_vectorizer.pkl")
tfidf_vec   = joblib.load(f"{MODELS_DIR}/tfidf_vectorizer.pkl")
lr_baseline = joblib.load(f"{MODELS_DIR}/lr_baseline.pkl")
lr_improved = joblib.load(f"{MODELS_DIR}/lr_improved.pkl")

print("All models loaded successfully.")

FileNotFoundError: [Errno 2] No such file or directory: 'models/count_vectorizer.pkl'

## 3 – Build feature matrices

In [ ]:
print("Building BoW features ...")
X_tr_bow  = get_bow_features(train_df, count_vec)
X_val_bow = get_bow_features(val_df,   count_vec)
X_te_bow  = get_bow_features(test_df,  count_vec)

print("Building improved features ...")
X_tr_all  = build_all_features(train_df, count_vec, tfidf_vec)
X_val_all = build_all_features(val_df,   count_vec, tfidf_vec)
X_te_all  = build_all_features(test_df,  count_vec, tfidf_vec)

print("Done.")

Building BoW features ...
Building improved features ...
Done.


## 4 – Evaluate baseline model

In [ ]:
baseline_results = {}
for split_name, X, df in [
    ("train", X_tr_bow,  train_df),
    ("val",   X_val_bow, val_df),
    ("test",  X_te_bow,  test_df),
]:
    y = df["is_duplicate"].values
    metrics = evaluate_model(lr_baseline, X, y, f"Baseline – {split_name}")
    baseline_results[split_name] = metrics

AttributeError: 'LogisticRegression' object has no attribute 'multi_class'

## 5 – Evaluate improved model

In [ ]:
improved_results = {}
for split_name, X, df in [
    ("train", X_tr_all,  train_df),
    ("val",   X_val_all, val_df),
    ("test",  X_te_all,  test_df),
]:
    y = df["is_duplicate"].values
    metrics = evaluate_model(lr_improved, X, y, f"Improved – {split_name}")
    improved_results[split_name] = metrics

## 6 – Summary DataFrame

In [ ]:
rows = []
for split, m in baseline_results.items():
    rows.append({"model": "Baseline (BoW + LR)", "split": split, **m})
for split, m in improved_results.items():
    rows.append({"model": "Improved (All features + LR)", "split": split, **m})

results_df = pd.DataFrame(rows)
results_df = results_df.round(4)
results_df